In [2]:
import requests
import json
import base64

from vantage6.client import UserClient

In [3]:
# This is standard authentication Keycloak flow. @Itziar; we need to discuss on how
# to deal with this for the demo. We could create a token that is valid for 10 years
# and use that token to authenticate?
client = UserClient(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es:443/server",
    auth_url="https://vantage6-auth.orchestrator.idea.lst.tfo.upm.es:443",
    auth_client="public_client",
    auth_realm="vantage6",
    log_level="INFO"
)
# You can authenticate using the user `itziar` and the password that I've send to you.
client.authenticate()

# Set the headers for the other requests
headers = {
    "Authorization": f"Bearer {client._access_token}"
}

# Print the server version
print("Server version: ", client.util.get_server_version())

 Welcome to
                  _                     __  
                 | |                   / /  
__   ____ _ _ __ | |_ __ _  __ _  ___ / /_  
\ \ / / _` | '_ \| __/ _` |/ _` |/ _ \ '_ \ 
 \ V / (_| | | | | || (_| | (_| |  __/ (_) |
  \_/ \__,_|_| |_|\__\__,_|\__, |\___|\___/ 
                            __/ |           
                           |___/            

 --> Join us on Discord! https://discord.gg/rwRvwyK
 --> Docs: https://docs.vantage6.ai
 --> Blog: https://vantage6.ai
------------------------------------------------------------
Cite us!
If you publish your findings obtained using vantage6, 
please cite the proper sources as mentioned in:
https://vantage6.ai/vantage6/references
------------------------------------------------------------
Opening browser for login


127.0.0.1 - - [17/Nov/2025 15:37:49] "GET /callback?state=state&session_state=6c052c7e-4b7f-483e-a6ef-89da733015c5&iss=https%3A%2F%2Fvantage6-auth.orchestrator.idea.lst.tfo.upm.es%2Frealms%2Fvantage6&code=c22a6c53-aead-4920-b5e6-0da5251a7d53.6c052c7e-4b7f-483e-a6ef-89da733015c5.fcb015fe-d5b0-4a7b-b609-7d87a3b72f3e HTTP/1.1" 200 -


 --> Succesfully authenticated
 --> Name: admin (id=1)
 --> Organization: root (id=1)
Server version:  {'version': '5.0.0a43'}


In [14]:
#
# Static content
#
image = "harbor2.vantage6.ai/idea4rc/sessions:latest"
method = "create_cohort"

# Organization IDs for the test collaboration with FAKE OMOP data
UPM_ORG_ID = 3
IKNL_ORG_ID = 1
ORG_IDS = [UPM_ORG_ID, IKNL_ORG_ID]

# Collaboration ID for the test collaboration with FAKE OMOP data
COLLABORATION_ID = 2

# Demo session ID
SESSION_ID = 2

# All organizations in the demo collaboration
STUDY_ID = 3

DATAFRAME_ID_PELVIS = 76 # Pelvis
DATAFRAME_ID_RPS_PELVIS = 77 # RPS+Pelvis
DATAFRAME_ID_RPS = 78 # RPS

#
# Dynamic content
#
RESULTS_COL = "sex"
GROUP_COLS = ["fnclcc_grade"]


org_input = [
    {
        "id": UPM_ORG_ID, # Central task is executed by UPM
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    "organizations_to_include": ORG_IDS
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

In [15]:
payload = {
    "name": "Human-readable name of the task",
    "image": "harbor2.vantage6.ai/idea4rc/analytics:latest",
    "description": "Description of the task",
    "action": "central_compute",
    "method": "crosstab",
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_RPS
            },
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_RPS_PELVIS
            },
            {
                "type": "dataframe",
                "dataframe_id": DATAFRAME_ID_PELVIS
            }
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}

In [22]:
# Create a vantage6 task to execute the summary analysis.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

{'job_id': 78,
 'algorithm_store': None,
 'description': 'Description of the task',
 'runs': '/server/run?task_id=218',
 'finished_at': None,
 'study': {'id': 3,
  'link': '/server/study/3',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'databases': [{'label': None,
   'type': 'dataframe',
   'dataframe_id': 78,
   'dataframe_name': 'RPS',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 76,
   'dataframe_name': 'Pelvis',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 77,
   'dataframe_name': 'Pelvis_RPS',
   'position': 0}],
 'dataframe': None,
 'required_by': [],
 'status': 'awaiting',
 'created_at': '2025-11-17T14:36:52.698518',
 'id': 218,
 'depends_on': [],
 'init_org': {'id': 1,
  'link': '/server/organization/1',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'parent': None,
 'collaboration': {'id': 2,
  'link': '/server/collaboration/2',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'children': '/server/task?parent_id=218',


In [25]:
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?job_id={JOB_ID}",
    headers=headers,
)
response.json()

{'data': [{'id': 249,
   'organization': {'id': 1,
    'link': '/server/organization/1',
    'methods': ['DELETE', 'GET', 'PATCH']},
   'task': {'id': 219,
    'link': '/server/task/219',
    'methods': ['GET', 'DELETE']},
   'action': 'federated_compute',
   'cleanup_at': None,
   'status': 'active',
   'blob_storage_used': False,
   'log': None,
   'node': {'status': 'online',
    'keycloak_client_id': '638fc96a-8954-4db4-92fa-09ff38421105',
    'name': 'Bilbao-root-node',
    'keycloak_id': '9768b637-e6bf-4f0c-b5bc-74c235856c47',
    'id': 7},
   'assigned_at': '2025-11-03T16:53:30.795454',
   'started_at': '2025-11-17T14:42:43.887071',
   'arguments': 'eyJyZXN1bHRzX2NvbCI6ICJzZXgiLCAiZ3JvdXBfY29scyI6IFsiZm5jbGNjX2dyYWRlIl19',
   'finished_at': None,
   'results': {'id': 249,
    'link': '/server/result/249',
    'methods': ['GET', 'PATCH']}},
  {'id': 248,
   'organization': {'id': 3,
    'link': '/server/organization/3',
    'methods': ['DELETE', 'GET', 'PATCH']},
   'task': {'id'